# 📨 Task 1 – Telegram Scraping & Data Lake Population  
📘 Version: 2025-07-14  

Structured data ingestion pipeline for **Kara Solutions’ Medical Business Intelligence Platform**. This notebook uses the Telethon API to scrape public Telegram channels related to Ethiopian medical businesses, storing raw message data and associated media (e.g., images) in a partitioned data lake. Outputs from this task form the raw layer of a modern ELT pipeline, powering downstream modeling (Task 2), enrichment (Task 3), and analytical APIs (Task 4).

---

**Challenge:** B5W7 – Shipping a Data Product  
**Company:** Kara Solutions  
**Author:** Nabil Mohamed  
**Branch:** `task-1-elt`  
**Date:** July 2025  

---

### 📌 This notebook covers:
- Authenticating with the Telegram API using Telethon  
- Scraping structured messages and downloading associated images  
- Organizing data into a partitioned file system (`data/raw/telegram_messages/YYYY-MM-DD/`)  
- Logging channel scrape status and handling API rate limits  
- Previewing raw data to verify scraping integrity  
- Preparing for JSON ingestion into PostgreSQL for dbt transformation


In [7]:
# ------------------------------------------------------------------------------
# 🛠 Ensure Notebook Runs from Project Root (for src/ imports to work)
# ------------------------------------------------------------------------------

import os
import sys

# If running from /notebooks/, move up to project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    print("📂 Changed working directory to project root")

# Add project root to sys.path so `src/` modules can be imported
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"✅ Added to sys.path: {project_root}")

# Optional: verify file presence to confirm we're in the right place
expected_path = "data/raw"
print(
    "📁 Output path ready"
    if os.path.exists(expected_path)
    else f"⚠️ Output path not found: {expected_path}"
)

📁 Output path ready


## 📦 Imports & Environment Setup

This cell loads the core libraries required for Telegram data scraping, partitioned file saving, logging, and initial data preview. Imports are grouped by function:

- **Telegram API access:** `telethon` for authentication and message scraping  
- **Data handling:** `json`, `pandas` for reading and previewing scraped message content  
- **Filesystem operations:** `os`, `pathlib` for dynamic folder creation and file storage  
- **Datetime utilities:** `datetime` for timestamp-based directory partitioning  
- **Logging & diagnostics:** `logging`, custom `logger` module for monitoring scrape runs  
- **Environment management:** `dotenv`, `config.py` for securely loading Telegram credentials  


In [4]:
# ---------------------------
# 📦 Imports & Environment Setup
# ---------------------------

# Telegram API
from telethon.sync import TelegramClient  # For interacting with Telegram API
from telethon.tl.functions.messages import GetHistoryRequest  # To fetch message history

# Data handling
import json  # For storing message data as JSON
import pandas as pd  # For previewing and exploring scraped data

# File system & time management
from pathlib import Path  # For OS-independent file path handling
from datetime import datetime  # For timestamp-based directory partitioning
import os  # For directory creation and file operations

# Logging & environment
import logging  # For runtime logging of scraping actions
from dotenv import load_dotenv  # For securely loading API keys from .env
import warnings  # To suppress Telethon or system-level warnings

# Configure pandas display options
pd.set_option("display.max_columns", None)  # Show all DataFrame columns
pd.set_option("display.float_format", "{:,.2f}".format)  # Consistent float formatting
warnings.filterwarnings("ignore")  # Suppress common warnings

## 📡 Telegram API Client Initialization

This section establishes a secure connection to the Telegram API using the `Telethon` library. Credentials (`TELEGRAM_API_ID`, `TELEGRAM_API_HASH`) are securely loaded from the `.env` file using `python-dotenv`.

The script validates the presence of required credentials and handles common connection errors gracefully. If successful, a `TelegramClient` session named `"kara_med_ingestor"` is initialized — ready to fetch message histories and media from public Telegram channels related to Ethiopian medical businesses.

In [ ]:
# ----------------------------------------
# 📡 Telegram API Client Initialization
# ----------------------------------------

from telethon.sync import TelegramClient  # For synchronous Telegram API interaction
from dotenv import load_dotenv  # For loading secrets from .env
import os  # For accessing environment variables

# Load environment variables from .env file
load_dotenv()  # Load TELEGRAM_API_ID and TELEGRAM_API_HASH

# Fetch credentials from environment
api_id = os.getenv("TELEGRAM_API_ID")  # Your Telegram API ID
api_hash = os.getenv("TELEGRAM_API_HASH")  # Your Telegram API Hash

# Validate presence of credentials
if not api_id or not api_hash:
    raise ValueError(
        "❌ Missing TELEGRAM_API_ID or TELEGRAM_API_HASH in your .env file."
    )

# Initialize the Telegram client under a persistent session name
try:
    client = TelegramClient("kara_med_ingestor", api_id, api_hash)  # Create session
    print("✅ Telegram client initialized.")
except Exception as e:
    print("❌ Failed to initialize Telegram client.")
    print("Error:", e)

# ----------------------------------------
# 🔌 Ensure Client is Connected Before Use
# ----------------------------------------

# Connect the client if not already connected
try:
    if not client.is_connected():  # Check if already connected
        await client.connect()  # Establish connection if needed
    print("🔌 Client connection status:", client.is_connected())  # Should print True
except Exception as e:
    print("❌ Failed to connect Telegram client.")
    print("Error:", e)

✅ Telegram client initialized and ready.


## 🧲 Channel Selection & Message Scraping

This section defines a curated list of public Telegram channels relevant to the Ethiopian medical and cosmetics sectors.

Using the `Telethon` API, the script fetches recent messages from each channel via the `GetHistoryRequest` method. For each message, the following metadata is extracted:
- Raw message content (`message`)
- Timestamp (`date`)
- Channel name and ID
- View count (if available)
- Media type (if an image or video is attached)

All results are stored as a list of dictionaries — structured for downstream export to a partitioned data lake (`data/raw/telegram_messages/YYYY-MM-DD/channel.json`) and optional preview in pandas for integrity checks.


## 🔐 Step 1: Request Telegram Login Code

To authorize the Telethon client for the first time, you'll need to log in with your Telegram account. This authentication step sends a one-time login code to your **Telegram app** (not via SMS).

**Instructions:**
- Enter your phone number in international format (e.g., `+2519XXXXXXXX`)
- Telegram will send a 5-digit login code via your **Telegram mobile app**
- In the next step, you'll enter this code to complete authorization
- After first-time login, the session will be saved and reused automatically (`kara_med_ingestor.session`)


In [20]:
# ------------------------------------------------------------------------------
# 🔐 Step 1: Request Login Code from Telegram
# ------------------------------------------------------------------------------

# Replace with your own phone number in international format
phone_number = "+251711029700"  # Example: +2519XXXXXXXX

# Send login code to your Telegram app (not via SMS)
try:
    await client.send_code_request(phone_number)  # Sends login code to your app
    print("✅ Login code sent successfully. Check your Telegram app.")
except Exception as e:
    print("❌ Failed to send login code. Please check the phone number and try again.")
    print("Error:", e)

✅ Login code sent successfully. Check your Telegram app.


## ✅ Step 2: Sign In Using the Code

Once you’ve received the 5-digit verification code via your Telegram app:

1. Replace `'12345'` in the next code cell with your actual verification code
2. Run the cell to complete your sign-in
3. A local session file (`kara_med_ingestor.session`) will be created and cached
   - You won’t need to re-authenticate in future runs unless you delete this file or revoke access


In [21]:
# ------------------------------------------------------------------------------
# ✅ Step 2: Sign In with Verification Code
# ------------------------------------------------------------------------------

# Replace this with the actual code sent to your Telegram app
verification_code = "63880"  # Example: 5-digit code received in-app

# Attempt to complete sign-in
try:
    await client.sign_in(phone_number, code=verification_code)  # Log in with code
    print("🎉 Authorization successful. You're now logged in.")
except Exception as e:
    print("❌ Login failed. Please verify the code and try again.")
    print("Error:", e)

🎉 Authorization successful. You're now logged in.


In [ ]:
# ------------------------------------------------------------------------------
# 📥 Scrape Messages + Download Media from Telegram (Task 1 — Full Pipeline + Logging + Retry)
# ------------------------------------------------------------------------------

import logging  # For structured logging
from telethon.tl.functions.messages import GetHistoryRequest  # For pulling messages
from telethon.errors import FloodWaitError  # To handle rate limits
from datetime import datetime  # For date partitioning
from pathlib import Path  # For structured storage
import json  # For writing raw message dumps
import asyncio  # For sleep handling during rate limits
import pandas as pd  # Optional preview/export

# -----------------------
# 📝 Logging Configuration
# -----------------------
today = datetime.today().strftime("%Y-%m-%d")  # Current date string

log_dir = Path("data/raw/scraping_logs")  # Directory for logs
log_dir.mkdir(parents=True, exist_ok=True)  # Create it if missing

log_file = log_dir / f"scraping_{today}.log"  # File path for today's log

# Reset logging handlers to avoid duplicates if re-running
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Set up logging to file + terminal
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler(log_file), logging.StreamHandler()],
)

logging.info("🚀 Starting Telegram scraping session...")

# -----------------------
# 🔌 Ensure Telegram Client is Connected
# -----------------------
await client.connect()
if not await client.is_user_authorized():
    logging.warning("🔐 Client not authorized. Please log in first.")
else:
    logging.info("✅ Telegram client is authorized and ready.")

# -----------------------
# 📡 Target Channels
# -----------------------
channel_usernames = [
    "lobelia4cosmetics",
    "tikvahpharma",
    "ChemedEthiopia",
]

# -----------------------
# 🔧 Scraping Settings
# -----------------------
total_limit = 500  # Messages per channel
batch_size = 100  # Telegram API max per call

# Global counters
total_scraped = 0
total_media = 0

# -----------------------
# 🔁 Loop Through Channels
# -----------------------
for username in channel_usernames:
    offset_id = 0
    collected = 0
    channel_messages = []

    logging.info(f"📡 Scraping channel: {username}")

    # Define output folders
    msg_dir = Path(f"data/raw/telegram_messages/{today}/{username}")
    img_dir = Path(f"data/raw/telegram_images/{today}/{username}")
    msg_dir.mkdir(parents=True, exist_ok=True)
    img_dir.mkdir(parents=True, exist_ok=True)

    while collected < total_limit:
        try:
            entity = await client.get_entity(username)  # Get channel object

            # Fetch batch of messages
            history = await client(
                GetHistoryRequest(
                    peer=entity,
                    limit=batch_size,
                    offset_date=None,
                    offset_id=offset_id,
                    max_id=0,
                    min_id=0,
                    add_offset=0,
                    hash=0,
                )
            )

            messages = history.messages
            if not messages:
                break

            for msg in messages:
                if not msg.message:
                    continue

                message_data = {
                    "channel": username,
                    "message": msg.message,
                    "views": msg.views,
                    "timestamp": msg.date.isoformat(),
                    "has_media": bool(msg.media),
                    "media_path": None,
                }

                if msg.media:
                    try:
                        media_path = await client.download_media(msg, file=img_dir)
                        if media_path:
                            message_data["media_path"] = str(media_path)
                            total_media += 1
                    except Exception as media_err:
                        logging.warning(f"⚠️ Media download failed: {media_err}")

                channel_messages.append(message_data)

            offset_id = messages[-1].id
            collected += len(messages)
            total_scraped += len(messages)

            logging.info(f"✅ {username}: {collected}/{total_limit} messages scraped")

        except FloodWaitError as fe:
            logging.warning(f"⏳ Rate limited. Sleeping for {fe.seconds} seconds...")
            await asyncio.sleep(fe.seconds)
        except Exception as e:
            logging.error(f"❌ Error scraping {username}: {e}")
            break

    # Save messages per channel
    output_path = msg_dir / f"{username}_{today}.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(channel_messages, f, ensure_ascii=False, indent=2)

    logging.info(
        f"📁 Saved {len(channel_messages)} messages to {output_path.resolve()}"
    )

# -----------------------
# 📊 Summary Log
# -----------------------
logging.info("🏁 Scraping complete.")
logging.info(f"🧾 Total channels scraped: {len(channel_usernames)}")
logging.info(f"🗂️  Total messages scraped: {total_scraped}")
logging.info(f"🖼️  Total media files downloaded: {total_media}")
logging.info(f"🪵 Log file saved to: {log_file.resolve()}")


📡 Scraping channel: lobelia4cosmetics
✅ lobelia4cosmetics: 100/500 messages collected
✅ lobelia4cosmetics: 200/500 messages collected
✅ lobelia4cosmetics: 300/500 messages collected
✅ lobelia4cosmetics: 400/500 messages collected
✅ lobelia4cosmetics: 500/500 messages collected
📁 Saved 482 messages to C:\Users\admin\Documents\GIT Repositories\b5w7-shipping-a-data-product-challenge\data\raw\telegram_messages\2025-07-15\lobelia4cosmetics\lobelia4cosmetics_2025-07-15.json

📡 Scraping channel: tikvahpharma
✅ tikvahpharma: 100/500 messages collected
✅ tikvahpharma: 200/500 messages collected
